# Week 11 Lab (extra): Hyperparameter Tuning

A random forest has many settings (*hyperparameters*) that you choose before training: how deep the trees may
grow, how many samples a leaf needs, how many features each split considers, and more. In this lab you'll tune
them two ways, with **grid search** and **random search**, and compare what each costs and what each finds.

This is good preparation for next week's classifier challenge: the data here is imbalanced, and we score
with **macro F1**, just as the challenge does.

**Core**: Exercises 1–4 (about 40 minutes). **Extension**: Exercise 5 (optional).

*Timing note:* the searches below fit many forests. On a 2-core Codespace, the grid search takes under a
minute and each random search about 10 seconds. `n_jobs=-1` uses all your cores.

Run this first: it imports everything and builds the data.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint, uniform
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV)

# 2,000 rows, 20 features (6 informative), and only about 15% positive cases
X, y = make_classification(n_samples=2000, n_features=20, n_informative=6, n_redundant=4,
                           weights=[0.85], flip_y=0.03, class_sep=0.8, random_state=407)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=407)

# The same 3 folds for every comparison, so differences come from the settings, not the split
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=407)

print(f"Training rows: {len(X_train)}, test rows: {len(X_test)}, share positive: {y_train.mean():.1%}")

#### **Exercise 1**

**Baseline.** Before tuning anything, measure a random forest with default settings (except
`n_estimators=50`, to keep things fast). Compute its mean **cross-validated** macro F1 on the training data
with `cross_val_score(..., cv=cv, scoring="f1_macro")`. Then fit it on all of the training data and report
macro F1 on the test set.

In [ ]:
rf = RandomForestClassifier(n_estimators=50, random_state=407)

# YOUR CODE HERE: cross-validated macro F1 on the training data (use cv=cv, scoring="f1_macro")
cv_scores = None
print(f"Cross-validated macro F1: {np.mean(cv_scores):.3f}")

# YOUR CODE HERE: fit rf on the training data, then compute macro F1 on the test set
test_f1 = None
print(f"Test macro F1: {test_f1:.3f}")

**Question:** we're about to try dozens of settings. Why should we compare them with cross-validation on the
training data rather than by their test-set score?

*Your answer here.*

#### **Exercise 2**

**Grid search.** `GridSearchCV` tries *every combination* of the values you list and keeps the one with the
best cross-validated score. Run it with the grid below, timing it. Then print the best settings, the best
CV score, and how many forests it fit in total (combinations × folds).

In [ ]:
param_grid = {
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 3, 10],
    "max_features": ["sqrt", 0.6],
    "class_weight": [None, "balanced"],
}

start = time.time()
# YOUR CODE HERE: create a GridSearchCV over RandomForestClassifier(n_estimators=50, random_state=407)
# using param_grid, cv=cv, scoring="f1_macro", n_jobs=-1; then fit it on the training data
grid = None
grid_seconds = time.time() - start

n_combos = len(grid.cv_results_["params"])
print(f"{n_combos} combinations x {cv.get_n_splits()} folds = {n_combos * cv.get_n_splits()} fits "
      f"in {grid_seconds:.0f} s")
print("Best settings:", grid.best_params_)
print(f"Best CV macro F1: {grid.best_score_:.3f}")

`grid.cv_results_` holds the score of every combination. The cell below averages the CV score over all
combinations that share each value of a hyperparameter. Run it, then answer: **which hyperparameter matters
most here, and which barely matters?**

In [ ]:
results = pd.DataFrame(grid.cv_results_)
for name in param_grid:
    col = "param_" + name
    summary = results.groupby(results[col].astype(str))["mean_test_score"].mean().round(3)
    print(f"{name}: {summary.to_dict()}")

*Your answer here.*

#### **Exercise 3**

**Random search.** `RandomizedSearchCV` doesn't try every combination. It draws `n_iter` random settings
from the ranges you give it. For numeric hyperparameters you can give a whole *distribution* instead of a
few values, so every draw tries a new value. Run it with `n_iter=10` (30 fits, versus the grid's 108), timing
it, and print the same three things.

In [ ]:
param_dist = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_leaf": randint(1, 15),       # any whole number from 1 to 14
    "max_features": uniform(0.1, 0.8),        # any fraction from 0.1 to 0.9
    "class_weight": [None, "balanced"],
}

start = time.time()
# YOUR CODE HERE: create a RandomizedSearchCV over RandomForestClassifier(n_estimators=50, random_state=407)
# using param_dist, n_iter=10, cv=cv, scoring="f1_macro", n_jobs=-1, random_state=0; then fit it
rand = None
rand_seconds = time.time() - start

print(f"{rand.n_iter} draws x {cv.get_n_splits()} folds = {rand.n_iter * cv.get_n_splits()} fits "
      f"in {rand_seconds:.0f} s")
print("Best settings:", rand.best_params_)
print(f"Best CV macro F1: {rand.best_score_:.3f}")

Random search is random, so one run could be lucky. Repeat it with `random_state` = 1, 2, 3, and 4 and
print each run's best CV score.

In [ ]:
# YOUR CODE HERE: loop over random_state in [1, 2, 3, 4], run the same RandomizedSearchCV, and print
# each run's best_score_

**Question:** how close does random search come to the grid's best score, for what share of the work?

*Your answer here.*

#### **Exercise 4**

**The test set, once.** Evaluate the grid search's and the (first) random search's best models on the test
set. (`best_estimator_` has already been refit on all of the training data.)

In [ ]:
# YOUR CODE HERE: test-set macro F1 for grid.best_estimator_ and rand.best_estimator_


**Question:** random search tried only 10 settings and did nearly as well. Why does it work so well? (Hint:
think about how many *different values* of the most important hyperparameters each method actually tried,
and look back at your Exercise 2 table.)

*Your answer here.*

#### **Exercise 5 (Extension)**

**More draws?** Run random search with `n_iter` = 5, 10, 20, and 40 (same `random_state`) and plot best CV
score against the number of fits. Where do the returns flatten out? Then try a small grid *around* the best
random-search settings (coarse-to-fine) and see whether it gains anything.

In [ ]:
# YOUR CODE HERE

*Your answer here.*